# Step 13 — MJO Preprocessing (Wheeler & Hendon 2004 method, simplified)
**Project:** ENSO-BSISO Self-Supervised Learning — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Preprocesses ERA5 u850, u200, OLR for the MJO project following **Wheeler & Hendon (2004)** with the simplifications agreed in Session 20:

| Step | Method | Decision |
|------|--------|----------|
| 1. Meridional average | average 15°S–15°N → shape `(T, 180)` per variable | **Q1 = A** |
| 2. Annual cycle | subtract mean + 3-harmonic Fourier (base period **1979–2001**, per WH04) | from paper |
| 3. Interannual variability | subtract preceding **120-day running mean** *only* (no SST1 regression) | **Q2 = B** |
| 4. Global variance normalization | divide each variable by its global temporal std | from paper |
| 5. Season scope | **all 12 months, 1979–2023** | **Q3 = A** |
| 6. Output shape | `(N, 3, 1, 180)` — Conv2d-compatible | **Q5 = recommendation** |

**Channel order:** `[u850, OLR, u200]` (matches the MJO extension plan)

**Inputs (from `MJO/data/raw/`):**
- `u850_u200_YYYY_YYYY.nc` × 5 chunks (from nb12)
- `OLR_MJO_1979_2023.nc` (from nb12)
- `rmm_labels.csv` (from nb11)

**Outputs (to `MJO/data/processed/`):**
- `X_MJO.npy` — shape `(N, 3, 1, 180)`, float32
- `labels_aligned_mjo.csv` — N rows, aligned to X_MJO
- `norm_stats_mjo.json` — global std per channel + metadata

---

## Cell 1 — Mount Google Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
RAW_DIR       = f'{MJO_DIR}/data/raw'
PROCESSED_DIR = f'{MJO_DIR}/data/processed'

os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Google Drive mounted.')
print(f'MJO raw files:')
for f in sorted(os.listdir(RAW_DIR)):
    mb = os.path.getsize(f'{RAW_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

## Cell 2 — Load ERA5 Wind (u850 + u200, 5 chunks)

In [ ]:
wind_files = sorted([
    f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
    if f.startswith('u850_u200') and f.endswith('.nc')
])

print(f'Found {len(wind_files)} wind files  (expected 45 annual files for 1979-2023):')
for f in wind_files[:5]:
    print(f'  {os.path.basename(f)}')
if len(wind_files) > 5:
    print(f'  ... and {len(wind_files)-5} more')

datasets = [xr.open_dataset(f) for f in wind_files]
ds_wind  = xr.concat(datasets, dim='valid_time').sortby('valid_time')

lats = ds_wind.latitude.values
lons = ds_wind.longitude.values
n_lat, n_lon = len(lats), len(lons)

wind_times = pd.DatetimeIndex(ds_wind.valid_time.values).normalize()

print(f'\nCombined wind: {len(wind_times)} days')
print(f'Date range:    {wind_times[0].date()} to {wind_times[-1].date()}')
print(f'Grid:          {n_lat} lat × {n_lon} lon')
print(f'Pressure lvls: {sorted(ds_wind.pressure_level.values.tolist())} hPa')
print(f'Lat range:     {lats.min():.1f} to {lats.max():.1f}')
print(f'Lon range:     {lons.min():.1f} to {lons.max():.1f}')

## Cell 3 — Load ERA5 OLR

In [ ]:
olr_files = sorted([
    f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
    if f.startswith('OLR_MJO_') and f.endswith('.nc')
])

print(f'Found {len(olr_files)} OLR files  (expected 45 annual files for 1979-2023)')

ds_olr    = xr.concat([xr.open_dataset(f) for f in olr_files],
                       dim='valid_time').sortby('valid_time')
olr_times = pd.DatetimeIndex(ds_olr.valid_time.values).normalize()

print(f'OLR combined: {len(olr_times)} days')
print(f'Date range:   {olr_times[0].date()} to {olr_times[-1].date()}')
print(f'Grid:         {len(ds_olr.latitude)} lat × {len(ds_olr.longitude)} lon')
print(f'Raw ttr range: [{float(ds_olr["ttr"].min()):.0f}, {float(ds_olr["ttr"].max()):.0f}] J/m²')

assert len(ds_olr.latitude) == n_lat, 'OLR/wind lat mismatch'
assert len(ds_olr.longitude) == n_lon, 'OLR/wind lon mismatch'

## Cell 4 — Load RMM Labels

In [ ]:
df_labels = pd.read_csv(f'{RAW_DIR}/rmm_labels.csv', parse_dates=['date'])
df_labels['date'] = df_labels['date'].dt.normalize()

print(f'Labels: {len(df_labels)} rows')
print(f'Date range: {df_labels["date"].min().date()} to {df_labels["date"].max().date()}')
print()
print('ENSO distribution:')
print(df_labels['enso_category'].value_counts())
print()
active = df_labels[~df_labels['weak_mjo']]
print(f'Active MJO days (ampl >= 1.0, phase 1-8): {len(active)}')
print(active['phase'].value_counts().sort_index())

## Cell 5 — Build Raw Arrays & Apply Meridional Average (Q1 = A)

Step 1 of WH04 preprocessing: meridionally average each variable over 15°S–15°N.  
This collapses the latitude axis → shape `(T, n_lon)` per variable.

In [ ]:
# Use intersection of wind and OLR dates (should be all dates)
common_dates = wind_times.intersection(olr_times).sort_values()
T = len(common_dates)
print(f'Common dates: {T}  (expected ~16,425 for 45 years all-month)')

wind_date_to_idx = {d: i for i, d in enumerate(wind_times)}
olr_date_to_idx  = {d: i for i, d in enumerate(olr_times)}
wind_idx = np.array([wind_date_to_idx[d] for d in common_dates])
olr_idx  = np.array([olr_date_to_idx[d]  for d in common_dates])

print('Loading u850 ...')
u850_3d = ds_wind['u'].sel(pressure_level=850).values[wind_idx].astype(np.float32)
print('Loading u200 ...')
u200_3d = ds_wind['u'].sel(pressure_level=200).values[wind_idx].astype(np.float32)
print('Loading OLR ...')
olr_3d  = (-ds_olr['ttr'].values[olr_idx]).astype(np.float32)

print(f'\nShapes before meridional average:')
print(f'  u850: {u850_3d.shape}')
print(f'  u200: {u200_3d.shape}')
print(f'  OLR:  {olr_3d.shape}')

# --- Meridional average over latitude axis (axis=1) ---
# WH04: 15°S to 15°N. Keep only lats within this band (inclusive).
lat_mask = (lats >= -15.0) & (lats <= 15.0)
print(f'\nLatitudes kept for meridional average: {lats[lat_mask]}  ({lat_mask.sum()} points)')

u850 = u850_3d[:, lat_mask, :].mean(axis=1).astype(np.float32)  # (T, n_lon)
u200 = u200_3d[:, lat_mask, :].mean(axis=1).astype(np.float32)
olr  = olr_3d[:,  lat_mask, :].mean(axis=1).astype(np.float32)

print(f'\nShapes after meridional average:')
print(f'  u850: {u850.shape}')
print(f'  u200: {u200.shape}')
print(f'  OLR:  {olr.shape}')
print(f'Memory: {3 * T * n_lon * 4 / 1e6:.1f} MB total')

# Free 3D arrays
del u850_3d, u200_3d, olr_3d, datasets

## Cell 6 — Step 2: Remove Annual Cycle (3 Fourier Harmonics, base 1979–2001)

WH04: subtract the climatological annual cycle (annual mean + first 3 harmonics) fitted over the base period 1979–2001 at each longitude.  
Same method as BSISO nb03 but base period matches the WH04 paper exactly.

In [ ]:
BASE_START  = 1979
BASE_END    = 2001
N_HARMONICS = 3
PERIOD      = 365

years     = common_dates.year.values
doys      = common_dates.day_of_year.values
base_mask = (years >= BASE_START) & (years <= BASE_END)

print(f'Base period: {BASE_START}-{BASE_END}')
print(f'Base period days available: {base_mask.sum()}')

def build_fourier_features(d, K=3, P=365):
    feats = [np.ones(len(d))]
    for k in range(1, K + 1):
        feats.append(np.cos(2 * np.pi * k * d / P))
        feats.append(np.sin(2 * np.pi * k * d / P))
    return np.column_stack(feats)

def remove_annual_cycle_harmonic(field_2d, doys, base_mask, K=3, P=365):
    """
    field_2d: (T, n_lon) float32
    Returns: (T, n_lon) anomaly after subtracting smooth annual cycle (K harmonics).
    """
    T, nx = field_2d.shape
    unique_doys = np.unique(doys)
    n_doys = len(unique_doys)

    # Step a: DOY climatology over base period → (n_doys, n_lon)
    clim = np.zeros((n_doys, nx), dtype=np.float64)
    for i, d in enumerate(unique_doys):
        ids = np.where((doys == d) & base_mask)[0]
        if len(ids) > 0:
            clim[i] = field_2d[ids].mean(axis=0)

    # Step b: fit K harmonics at every longitude
    X_fit  = build_fourier_features(unique_doys, K, P)
    coeffs, _, _, _ = np.linalg.lstsq(X_fit, clim, rcond=None)

    # Step c: evaluate smooth cycle for every observed day and subtract
    X_all  = build_fourier_features(doys, K, P)
    smooth = (X_all @ coeffs).astype(np.float32)
    return field_2d - smooth

print(f'Removing {N_HARMONICS}-harmonic annual cycle from u850 ...')
u850_a = remove_annual_cycle_harmonic(u850, doys, base_mask, N_HARMONICS, PERIOD)
print(f'Removing {N_HARMONICS}-harmonic annual cycle from u200 ...')
u200_a = remove_annual_cycle_harmonic(u200, doys, base_mask, N_HARMONICS, PERIOD)
print(f'Removing {N_HARMONICS}-harmonic annual cycle from OLR  ...')
olr_a  = remove_annual_cycle_harmonic(olr,  doys, base_mask, N_HARMONICS, PERIOD)

print(f'\nPost-step-1 mean over base period (should be ~0):')
print(f'  u850: {u850_a[base_mask].mean():+.6f}')
print(f'  u200: {u200_a[base_mask].mean():+.6f}')
print(f'  OLR:  {olr_a[base_mask].mean():+.6f}')

## Cell 7 — Step 3: Remove Interannual Variability (120-day Running Mean, Q2 = B)

Simplified Lee approach: subtract the preceding 120-day running mean only (no SST1 regression).  
Because we have all-year data, the full 120-day window is satisfied for every day after early May 1979.

In [ ]:
WINDOW = 120

def remove_running_mean(anom_2d, dates, window=120):
    """
    Subtract preceding `window`-day running mean per longitude.
    anom_2d: (T, n_lon) float32, dates: DatetimeIndex of length T
    """
    T, nx = anom_2d.shape
    df = pd.DataFrame(anom_2d.astype(np.float64), index=dates)
    rm = df.rolling(window=window, min_periods=1, closed='left').mean().values
    return (anom_2d - rm).astype(np.float32)

print('Removing 120-day preceding running mean from u850 ...')
u850_iso = remove_running_mean(u850_a, common_dates, WINDOW)
print('Removing 120-day preceding running mean from u200 ...')
u200_iso = remove_running_mean(u200_a, common_dates, WINDOW)
print('Removing 120-day preceding running mean from OLR  ...')
olr_iso  = remove_running_mean(olr_a,  common_dates, WINDOW)

# Window coverage check
print(f'\nFirst 5 days running-mean window coverage:')
for i in range(5):
    avail = min(i, WINDOW)
    print(f'  {common_dates[i].date()}: {avail}/{WINDOW} ({avail/WINDOW*100:.0f}%)')
print(f'  ... after day {WINDOW}, every day has full 120-day history.')

## Cell 8 — Step 4: Global Variance Normalization

WH04 step 3: divide each variable by its global (all-longitude, all-time) temporal standard deviation over the base period.  
After meridional averaging, each variable is a 2D `(T, n_lon)` field — we compute one scalar std over all entries in the base period.

In [ ]:
def global_std_normalize(iso_2d, base_mask):
    """
    Compute one scalar std over all (time, longitude) entries in base period,
    then divide the full field by it.
    """
    base_vals  = iso_2d[base_mask].ravel()
    scalar_std = float(base_vals.std())
    return (iso_2d / scalar_std).astype(np.float32), scalar_std

u850_n, std_u850 = global_std_normalize(u850_iso, base_mask)
u200_n, std_u200 = global_std_normalize(u200_iso, base_mask)
olr_n,  std_olr  = global_std_normalize(olr_iso,  base_mask)

norm_stats = {
    'method'              : 'global_temporal_std (WH04)',
    'base_period'         : f'{BASE_START}-{BASE_END}',
    'n_harmonics'         : N_HARMONICS,
    'running_mean_window' : WINDOW,
    'meridional_band'     : '15S-15N',
    'enso_removal'        : 'simplified Lee (120-day running mean only, no SST1)',
    'channel_order'       : ['u850', 'OLR', 'u200'],
    'u850' : {'global_std': std_u850},
    'OLR'  : {'global_std': std_olr},
    'u200' : {'global_std': std_u200},
}

print('Global temporal std (base period):')
print(f'  u850: {std_u850:.4f} m/s')
print(f'  u200: {std_u200:.4f} m/s')
print(f'  OLR:  {std_olr:.4f} J/m²')

print(f'\nPost-normalization std over base period (should be ~1):')
print(f'  u850: {u850_n[base_mask].std():.4f}')
print(f'  u200: {u200_n[base_mask].std():.4f}')
print(f'  OLR:  {olr_n[base_mask].std():.4f}')

## Cell 9 — Stack Channels & Align with Labels

Channel order: `[u850, OLR, u200]` as specified in the MJO extension plan.  
Final shape: `(N, 3, 1, 180)` — the singleton lat axis keeps the array Conv2d-compatible (Q5).

In [ ]:
label_dates = pd.DatetimeIndex(df_labels['date'])
aligned     = common_dates.intersection(label_dates).sort_values()

print(f'ERA5 dates:          {len(common_dates)}')
print(f'Label dates:         {len(label_dates)}')
print(f'Aligned (both):      {len(aligned)}')
print(f'Date range:          {aligned[0].date()} to {aligned[-1].date()}')

common_date_to_idx = {d: i for i, d in enumerate(common_dates)}
sel_idx = np.array([common_date_to_idx[d] for d in aligned])

# Stack as (N, 3, 1, 180): channels = [u850, OLR, u200]; lat-axis is singleton
X_MJO = np.stack([
    u850_n[sel_idx],
    olr_n[sel_idx],
    u200_n[sel_idx],
], axis=1)[:, :, np.newaxis, :].astype(np.float32)

print(f'\nX_MJO shape: {X_MJO.shape}  (expect (~16425, 3, 1, 180))')
print(f'Memory:      {X_MJO.nbytes / 1e6:.1f} MB')

# Align labels
df_aligned = df_labels[df_labels['date'].isin(aligned)].copy()
df_aligned = df_aligned.sort_values('date').reset_index(drop=True)

assert list(df_aligned['date']) == list(aligned), 'Date order mismatch!'
print(f'\nLabels aligned: {len(df_aligned)} rows')
print(df_aligned[['date', 'rmm1', 'rmm2', 'phase', 'amplitude', 'enso_category', 'weak_mjo']].head(5))

## Cell 10 — Save Outputs

In [ ]:
X_path     = f'{PROCESSED_DIR}/X_MJO.npy'
y_path     = f'{PROCESSED_DIR}/labels_aligned_mjo.csv'
stats_path = f'{PROCESSED_DIR}/norm_stats_mjo.json'
lon_path   = f'{PROCESSED_DIR}/longitudes_mjo.npy'

np.save(X_path, X_MJO)
df_aligned.to_csv(y_path, index=False)
np.save(lon_path, lons.astype(np.float32))
with open(stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)

print('Saved:')
for p in [X_path, y_path, stats_path, lon_path]:
    mb = os.path.getsize(p) / 1e6
    print(f'  {os.path.basename(p):30s}  ({mb:.2f} MB)')

## Cell 11 — Verification: RMM Phase Composites (Longitude profiles)

Expected MJO signature: OLR' anomaly (active convection) propagates **eastward** through phases 1→8:
- Phase 1: enhanced convection over Indian Ocean (~80°E)
- Phase 2-3: Maritime Continent (~100-130°E)
- Phase 4-5: West Pacific (~140-160°E)
- Phase 6-8: signal weakens, returns to West Hemisphere

In [ ]:
import matplotlib.pyplot as plt

active = df_aligned[~df_aligned['weak_mjo'] & (df_aligned['phase'].between(1, 8))]

fig, axes = plt.subplots(8, 1, figsize=(14, 12), sharex=True)
fig.suptitle('MJO Phase Composites — OLR (red dashed) and u850 (solid blue)  [meridionally averaged 15°S–15°N]',
             fontsize=12, fontweight='bold')

lon_axis = lons
for ax, ph in zip(axes, range(1, 9)):
    idx = active[active['phase'] == ph].index.values
    if len(idx) == 0:
        ax.set_title(f'Phase {ph} — no data')
        continue
    olr_comp  = X_MJO[idx, 1, 0, :].mean(axis=0)
    u850_comp = X_MJO[idx, 0, 0, :].mean(axis=0)

    ax.plot(lon_axis, u850_comp, 'b-', lw=1.5, label='u850 anom')
    ax.plot(lon_axis, olr_comp,  'r--', lw=1.5, label='OLR anom')
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_ylabel(f'Phase {ph}\n(N={len(idx)})', fontsize=9)
    ax.grid(True, alpha=0.3)
    if ph == 1:
        ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Longitude (°)', fontsize=11)
plt.tight_layout()
fig_path = f'{PROCESSED_DIR}/mjo_phase_composites.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('Look for the OLR minimum (red dashed dip) shifting eastward as phase increases.')

## Cell 12 — Verification: Zonal Hovmöller Diagram (one MJO event)

Plots OLR' anomaly as time-longitude for a year known to have a strong MJO event.  
Eastward propagation appears as a tilted band of negative OLR' anomaly from west to east.

In [ ]:
# Pick a year with strong MJO activity. The 1992-93 MJO was canonical.
HOV_START = pd.Timestamp('1992-11-01')
HOV_END   = pd.Timestamp('1993-03-31')

date_arr = pd.DatetimeIndex(df_aligned['date'])
hov_mask = (date_arr >= HOV_START) & (date_arr <= HOV_END)
hov_idx  = np.where(hov_mask)[0]

olr_hov  = X_MJO[hov_idx, 1, 0, :]   # (T_hov, n_lon)
u850_hov = X_MJO[hov_idx, 0, 0, :]
hov_dates = date_arr[hov_idx]

print(f'Hovmöller window: {hov_dates[0].date()} to {hov_dates[-1].date()}  ({len(hov_idx)} days)')

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.suptitle(f'Zonal Hovmöller (15°S–15°N average)  |  {HOV_START.date()} to {HOV_END.date()}',
             fontsize=12, fontweight='bold')

for ax, data, title, cmap in zip(
    axes,
    [olr_hov, u850_hov],
    ["OLR' anomaly (negative = convection)", "u850' anomaly"],
    ['RdBu_r', 'RdBu_r'],
):
    vmax = float(np.percentile(np.abs(data), 95))
    im = ax.imshow(data, cmap=cmap, aspect='auto',
                   extent=[lons.min(), lons.max(), len(hov_idx), 0],
                   vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude (°)')
    ax.set_ylabel('Days from start')
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label='σ')

plt.tight_layout()
fig_path = f'{PROCESSED_DIR}/mjo_hovmoller_1992_93.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('Look for diagonally-tilted bands sloping from upper-left to lower-right → eastward propagation.')

## Cell 13 — Verification: ENSO Composites (should be ~0)

After Lee preprocessing, ENSO-related slow variability should be largely removed by the 120-day running mean.  
Any residual is potentially what the SSL encoder will pick up (the ENSO-modulated *intraseasonal* signal).

In [ ]:
enso_cats = ['El Nino', 'Neutral', 'La Nina']
fig, axes = plt.subplots(3, 2, figsize=(15, 9))
fig.suptitle('ENSO composites (OLR & u850) — should be near-zero after Lee preprocessing',
             fontsize=12, fontweight='bold')

vmax_olr  = float(np.percentile(np.abs(X_MJO[:, 1, 0, :]), 90))
vmax_u850 = float(np.percentile(np.abs(X_MJO[:, 0, 0, :]), 90))

for row, cat in enumerate(enso_cats):
    idx = df_aligned[df_aligned['enso_category'] == cat].index.values
    if len(idx) == 0:
        for col in range(2):
            axes[row, col].set_title(f'{cat} — no data')
        continue
    olr_comp  = X_MJO[idx, 1, 0, :].mean(axis=0)
    u850_comp = X_MJO[idx, 0, 0, :].mean(axis=0)

    axes[row, 0].plot(lons, olr_comp, 'r-', lw=1.5)
    axes[row, 0].axhline(0, color='k', lw=0.5, alpha=0.5)
    axes[row, 0].set_ylim(-vmax_olr, vmax_olr)
    axes[row, 0].set_ylabel(f'{cat}\n(N={len(idx)})', fontsize=9)
    axes[row, 0].grid(True, alpha=0.3)
    axes[row, 0].set_title(f"OLR' composite  max|x|={np.abs(olr_comp).max():.3f}σ", fontsize=9)

    axes[row, 1].plot(lons, u850_comp, 'b-', lw=1.5)
    axes[row, 1].axhline(0, color='k', lw=0.5, alpha=0.5)
    axes[row, 1].set_ylim(-vmax_u850, vmax_u850)
    axes[row, 1].grid(True, alpha=0.3)
    axes[row, 1].set_title(f"u850' composite  max|x|={np.abs(u850_comp).max():.3f}σ", fontsize=9)

axes[-1, 0].set_xlabel('Longitude (°)')
axes[-1, 1].set_xlabel('Longitude (°)')
plt.tight_layout()
fig_path = f'{PROCESSED_DIR}/mjo_enso_composites.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print('Interpretation: small |composite| (≪ 0.5σ) → background removed; large → leftover ENSO signal.')

## Cell 14 — Summary Report

In [ ]:
print('=' * 65)
print('MJO PREPROCESSING SUMMARY')
print('=' * 65)
print(f'Method:               Wheeler & Hendon (2004), simplified')
print(f'Input data:           ERA5 all-year, 1979-2023, 15°S-15°N global strip')
print(f'Base period:          {BASE_START}-{BASE_END}')
print(f'Annual cycle:         {N_HARMONICS}-harmonic Fourier (per longitude)')
print(f'ENSO removal:         {WINDOW}-day preceding running mean (Q2 = B, no SST1)')
print(f'Normalization:        global temporal std (one scalar per variable)')
print()
print(f'Output array:         X_MJO.npy  shape={X_MJO.shape}  float32')
print(f'Channel order:        [u850, OLR, u200]')
print(f'Labels:               labels_aligned_mjo.csv  {len(df_aligned)} rows')
print()
print('Normalization scalars (global temporal std, base period):')
print(f'  u850: {std_u850:.4f} m/s')
print(f'  u200: {std_u200:.4f} m/s')
print(f'  OLR:  {std_olr:.4f} J/m²')
print()
print('Label coverage:')
print(f'  Total days:                {len(df_aligned)}')
print(f'  Active MJO (ampl >= 1.0):  {(~df_aligned["weak_mjo"]).sum()}')
print(f'  ENSO  El Nino / Neutral / La Nina:  '
      f'{(df_aligned["enso_category"]=="El Nino").sum()} / '
      f'{(df_aligned["enso_category"]=="Neutral").sum()} / '
      f'{(df_aligned["enso_category"]=="La Nina").sum()}')
print()
print('Next: nb14 (supervised 2D) and nb15 (SSL temporal 2D)')
print('=' * 65)

---
## Done!

Google Drive should now contain:

```
BSISO_SSL_Project/MJO/data/processed/
├── X_MJO.npy                    (N, 3, 1, 180) float32
├── labels_aligned_mjo.csv       N rows aligned to X_MJO
├── norm_stats_mjo.json          metadata + global stds
├── longitudes_mjo.npy           (180,) for plotting
├── mjo_phase_composites.png
├── mjo_hovmoller_1992_93.png
└── mjo_enso_composites.png
```

**Next:**
- **nb14** (`14_mjo_supervised_2d.ipynb`) — supervised CNN with RMM phase + ENSO pair labels
- **nb15** (`15_mjo_ssl_temporal_2d.ipynb`) — SSL temporal encoder with 20–90 day bandpass

---
*DDCS Project | jh9141@nyu.edu*